# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step walkthrough for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library, referencing all fields by their `@id` values.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata (not as a dict, via properties)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")


## 2. Data Overview
Review available record sets (`@id` values), their fields, and column `@id` values using the dataset metadata. All references are by `@id`.

In [ ]:
# List all record sets by their @id
print("\nAvailable record sets (by @id):")
record_sets = dataset.metadata.record_set
if not record_sets:
    print('No record sets found in metadata. Trying dataset.record_sets property.')
    # Fallback: try to collect record sets from the API, even if not present in metadata
    record_sets = list(dataset.record_sets)

# In practice, mlcroissant parses record_sets lazily, so safest way:
record_set_ids = []
for record_set in dataset.record_sets:
    print(f"- @id: {record_set['@id']}")
    record_set_ids.append(record_set['@id'])
    name = record_set.get('name') or record_set.get('rdfs:label')
    if name:
        print(f"  name: {name}")
    # List field @ids (columns)
    columns = record_set.get('cr:column', [])
    if isinstance(columns, dict):
        columns = [columns]
    print("  columns:")
    for col in columns:
        colid = col.get('@id', col)
        print(f"    - {colid}")


## 3. Data Extraction
Load data from the main record set(s) into pandas DataFrames. Use the record set and field `@id`s listed above.

In [ ]:
# For this dataset, there is only one main tabular record set 
# We'll extract all data using the identified record set @id

# Set the record set ids explicitly here after inspection above
# If more than one, you can list them all here
# Example: record_set_ids = ['https://sen.science/doi/10.71728/senscience.qs2f-h81p/record-set']
record_set_ids = []
for record_set in dataset.record_sets:
    record_set_ids.append(record_set['@id'])

# Extract data from each record set into a pandas DataFrame
dataframes = {}
for rsid in record_set_ids:
    print(f'Loading records for record_set @id: {rsid}')
    # List of records (list of dicts)
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)
    print(f'Columns for {rsid}:', dataframes[rsid].columns.tolist())
    display(dataframes[rsid].head(3))

# Choose the first/main record set for analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None


## 4. Exploratory Data Analysis (EDA)
Explore and process the data using field and column `@id`s. This includes selecting numeric fields, filtering, normalization, and grouping.

In [ ]:
from pandas.api.types import is_numeric_dtype
import numpy as np

# Display all column @id values in the main dataframe
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    print("Main DataFrame columns (@id):\n", df.columns.tolist())
    
    # Attempt to find a numeric field automatically (since @id can be verbose)
    numeric_field_id = None
    for c in df.columns:
        if is_numeric_dtype(df[c]):
            numeric_field_id = c
            break
        # Try to coerce non-numeric columns if possible
    if not numeric_field_id:
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
                if is_numeric_dtype(df[c]):
                    numeric_field_id = c
                    break
            except Exception:
                pass
    if numeric_field_id:
        print(f"Chosen numeric field (@id): {numeric_field_id}")
    else:
        print('No numeric fields found. Skipping filtering/normalization.')

    # Set threshold for filtering demo
    threshold = None
    if numeric_field_id:
        # Use the median as an example threshold (or 10, if plausible)
        median_val = df[numeric_field_id].median()
        threshold = median_val
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold} (median): {len(filtered_df)} records")
        display(filtered_df.head(3))

        # Normalize column
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"\nNormalized {numeric_field_id} (z-score normalization):")
        display(filtered_df[[numeric_field_id, norm_col]].head(3))

        # Try to find a categorical/grouping field
        group_field_id = None
        # Pick the first 'object'-type column that is not the numeric field
        for c in df.columns:
            if c == numeric_field_id:
                continue
            if df[c].dtype == object or df[c].dtype.name.startswith('category'):
                group_field_id = c
                break
        if group_field_id:
            print(f"\nGrouping by field (@id): {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean of {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
        else:
            print('No suitable grouping field found.')
else:
    print('No available record set for EDA.')


## 5. Visualization
Visualize the distribution of the chosen numeric field and groupwise means using matplotlib. All axes are labeled using field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if main_record_set_id is not None and numeric_field_id:
    # Histogram of numeric field
    plt.figure(figsize=(6, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=10)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Bar plot of groupwise mean if grouped_df exists
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(8,4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ci=None)
        plt.title(f'Mean {numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(f'Mean {numeric_field_id}')
        plt.xticks(rotation=30)
        plt.show()
else:
    print('Numeric field or record set unavailable for visualization.')


## 6. Conclusion
We have loaded and explored the FAIR² colorectal cancer dataset using the `mlcroissant` library, referencing all entities by their `@id`. Data fields and record sets were identified by inspecting the metadata, numeric fields were normalized and analyzed, and exploratory visualizations were produced. For more advanced workflows, continue using the explicit `@id` references for robust and reproducible data analysis pipelines.